[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sunilmogadati/systems-in-production/blob/main/implementation/notebooks/Deep_Learning_Autoencoders_GANs.ipynb)

Playbook: [AI / Deep Learning](../../playbooks/ai/deep-learning/README.md)


# AI Engineer Accelerator: Autoencoders and GANs Deep Dive

**From Compression to Creation — Autoencoders and Generative Adversarial Networks**

| | |
|:---|:---|
| **Program** | [Emergtech AI Engineer Accelerator](https://github.com/sunilmogadati/systems-in-production) |
| **Author** | Sunil Mogadati |
| **Week** | 3-4 — Deep Learning: Unsupervised Learning |
| **Also serves as** | CSCI 5931 Deep Learning — Exam 2 Study Guide (Autoencoder section) |



## What You Will Learn

- The paradigm shift from supervised to unsupervised learning
- How autoencoders learn to compress and reconstruct data without labels
- Why the bottleneck forces genuine feature learning
- Types of autoencoders: undercomplete, denoising, deep, convolutional
- How to build autoencoders in PyTorch and Keras
- Real-world applications: compression, denoising, anomaly detection, colorization
- **GANs: how two competing networks learn to generate realistic data**
- **The generator/discriminator architecture and adversarial training loop**
- **SimpleGAN vs DCGAN — fully connected vs convolutional generation**
- **The minimax objective function and GAN training dynamics**


## Key Terms

| Term | Plain English |
|:---|:---|
| **Unsupervised Learning** | Learning from data without labels — discover structure on your own |
| **Autoencoder** | Neural network that learns to compress input and reconstruct it; target = input |
| **Encoder** | The compression half — maps input x to a smaller code h |
| **Decoder** | The reconstruction half — maps code h back to an approximation of x |
| **Bottleneck / Code / Latent** | The compressed representation between encoder and decoder |
| **Reconstruction Loss** | How different is the output from the input? L(x, g(f(x))) |
| **Undercomplete** | Bottleneck smaller than input — forces compression |
| **Overcomplete** | Bottleneck larger than input — can cheat by copying |
| **Denoising AE** | Trained on noisy input, reconstructs clean original |
| **Convolutional AE** | Uses Conv2d/UpSampling instead of dense layers — preserves spatial structure |


---
## Part I: Concept and Mental Model


## 1. The Paradigm Shift — From Supervised to Unsupervised

Everything so far has been **supervised learning:** given input X and target Y, learn X→Y.

| Paradigm | Data | Task | Example |
|:---|:---|:---|:---|
| **Supervised** | X + Y (labeled) | Predict Y from X | Classify MNIST digits |
| **Unsupervised** | X only (no labels) | Discover structure in X | Compress, cluster, generate |

*Analogy:* Supervised learning = teacher giving you flashcards with questions AND answers. Unsupervised learning = giving you a pile of photographs with no labels. You must discover patterns on your own.


## 2. What Is an Autoencoder?

### The Mental Model

*A game of telephone with a purpose.* Person A (encoder) must describe a complex photograph using only 10 words (the bottleneck). Person B (decoder) must recreate the photograph from those 10 words. If the reconstruction is close, the 10 words captured the **essence** of the image. Those 10 words ARE the learned features.

```
Input x (784 pixels) → [Encoder f(x)] → Code h (32 numbers) → [Decoder g(h)] → Reconstruction r (784 pixels)
```

**The key insight:** You train the network to reconstruct its own input (Y = X). The bottleneck forces it to compress — it can't simply copy every pixel. It must learn **which features matter enough to survive the bottleneck.**

**Loss function:** L(x, r) = L(x, g(f(x))) — how different is the output from the input?

### Why Does This Work?

If the bottleneck dimension (32) is smaller than the input (784), the network MUST learn a compressed representation. It's mathematically related to PCA when encoder/decoder are linear. But with nonlinear activations, autoencoders capture far more powerful compressions — nonlinear manifolds in the data.


## 3. Hello World — Simple MNIST Autoencoder


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# === DATA ===
transform = transforms.Compose([transforms.ToTensor()])
train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_data = datasets.MNIST(root='./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=128, shuffle=False)

# === AUTOENCODER: Simplest possible ===
class SimpleAutoencoder(nn.Module):
    """
    Input (784) -> Encoder (128 -> 32) -> Decoder (128 -> 784)
    
    The bottleneck is 32 dimensions.
    784 -> 32 = 24.5x compression.
    """
    def __init__(self):
        super().__init__()
        # Encoder: compress 784 -> 32
        self.encoder = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 32),   # BOTTLENECK: 32 dimensions
            nn.ReLU()
        )
        # Decoder: reconstruct 32 -> 784
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid()          # Output in [0,1] to match pixel values
        )
    
    def forward(self, x):
        x = x.view(-1, 784)      # Flatten
        code = self.encoder(x)    # Compress
        reconstruction = self.decoder(code)  # Reconstruct
        return reconstruction

model = SimpleAutoencoder()
print(model)
print(f'\nBottleneck: 784 -> 32 ({784/32:.1f}x compression)')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')


In [ ]:
# === TRAIN: Target IS the input! ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleAutoencoder().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()  # Pixel-level reconstruction loss

for epoch in range(10):
    total_loss = 0
    for images, _ in train_loader:    # We IGNORE labels (_) — unsupervised!
        images = images.to(device)
        images_flat = images.view(-1, 784)
        
        optimizer.zero_grad()
        reconstruction = model(images)
        loss = criterion(reconstruction, images_flat)  # Compare reconstruction to ORIGINAL
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f'Epoch {epoch+1}: Loss={total_loss/len(train_loader):.6f}')

print('\nNotice: model.fit(x_train, x_train) — target IS the input!')


In [ ]:
# === VISUALIZE: Original vs Reconstruction ===
model.eval()
with torch.no_grad():
    test_images, _ = next(iter(test_loader))
    test_images = test_images.to(device)
    reconstructions = model(test_images).view(-1, 1, 28, 28).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(test_images[i].cpu().squeeze(), cmap='gray')
    axes[0, i].set_title('Original')
    axes[0, i].axis('off')
    axes[1, i].imshow(reconstructions[i].squeeze(), cmap='gray')
    axes[1, i].set_title('Reconstructed')
    axes[1, i].axis('off')
plt.suptitle('Simple Autoencoder: 784 -> 32 -> 784')
plt.tight_layout()
plt.show()
print('Reconstructions are slightly blurry — the bottleneck forces lossy compression')


## 4. Why Does This Work?

**The bottleneck creates an information bottleneck.** 784 pixels → 32 numbers means 96% of the information must be discarded. The network learns to keep the 4% that matters most for reconstruction.

What does the 32-dimensional code capture?
- Not individual pixels (too many to memorize in 32 numbers)
- Instead: stroke patterns, curves, loops, line directions — the **essence** of each digit
- Digit '1' has a simple vertical stroke → easy to encode in few numbers
- Digit '8' has two loops → needs more code capacity

**Connection to PCA:** A linear autoencoder (no activation functions) learns the same subspace as PCA — the directions of maximum variance. With nonlinear activations, autoencoders learn curved manifolds that PCA can't capture.


---
## Part II: Types of Autoencoders


## 5. Undercomplete vs Overcomplete

**Undercomplete (standard):** Bottleneck < input dimension
- Forces genuine compression
- The encoding IS the learned feature representation
- This is what we built in Section 3

**Overcomplete (dangerous):** Bottleneck >= input dimension
- Can 'cheat' by simply copying the input through unchanged
- Must add constraints (regularization) to prevent trivial identity mapping
- Denoising autoencoders solve this problem


## 6. Denoising Autoencoder

### The Mental Model

*Learning to read handwriting with coffee stains on it. If you can reconstruct the clean text from the stained version, you've truly learned the STRUCTURE of the writing, not just memorized the stains.*

**How it works:**
1. Take clean input x
2. Corrupt it: x_noisy = x + noise
3. Feed x_noisy to the autoencoder
4. Loss = difference between output and **clean** x (not noisy x_noisy!)

The network must learn robust features that survive noise — it can't just memorize pixel positions.


In [ ]:
class DenoisingAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(),
            nn.Linear(256, 64), nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(),
            nn.Linear(256, 784), nn.Sigmoid()
        )
    
    def forward(self, x):
        return self.decoder(self.encoder(x.view(-1, 784)))

# Training: add noise to INPUT, compare to CLEAN target
def train_denoising(model, train_loader, epochs=10, noise_factor=0.3):
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()
    device = next(model.parameters()).device
    
    for epoch in range(epochs):
        total_loss = 0
        for images, _ in train_loader:
            clean = images.view(-1, 784).to(device)
            # Add Gaussian noise
            noisy = clean + noise_factor * torch.randn_like(clean)
            noisy = torch.clamp(noisy, 0., 1.)  # Keep in valid range
            
            optimizer.zero_grad()
            output = model(noisy)
            loss = criterion(output, clean)  # Compare to CLEAN, not noisy!
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch+1}: Loss={total_loss/len(train_loader):.6f}')

print('Denoising AE: feed noisy input, reconstruct clean output')
print('Loss compares to CLEAN original, not the noisy version')


## 7. Deep Autoencoder

Multiple layers in both encoder and decoder — captures hierarchical features.

```python
# Mirror architecture: encoder layers reversed in decoder
Encoder: 784 -> 512 -> 256 -> 128 -> 64 -> 32   (progressively compress)
Decoder: 32 -> 64 -> 128 -> 256 -> 512 -> 784   (progressively expand)
```

Deeper = more expressive. But also harder to train. Use ReLU activations and consider batch normalization.


## 8. Convolutional Autoencoder

### The Mental Model

Dense autoencoders flatten images — losing spatial structure (same problem as FC networks for classification). Convolutional autoencoders **preserve the 2D structure**:

- **Encoder:** Conv2d + MaxPooling (downsample — like CNN feature extraction)
- **Decoder:** Conv2d + UpSampling2d (upsample — the reverse operation)

```
Encoder: 28x28x1 → Conv → Pool → 14x14x16 → Conv → Pool → 7x7x8  (compress)
Decoder: 7x7x8 → Conv → UpSample → 14x14x16 → Conv → UpSample → 28x28x1  (expand)
```


In [ ]:
class ConvAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder: downsample with conv + pool
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1),   # 28x28x1 -> 28x28x16
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                # 28x28x16 -> 14x14x16
            nn.Conv2d(16, 8, 3, padding=1),   # 14x14x16 -> 14x14x8
            nn.ReLU(),
            nn.MaxPool2d(2, 2)                 # 14x14x8 -> 7x7x8
        )
        # Decoder: upsample with conv + upsample
        self.decoder = nn.Sequential(
            nn.Conv2d(8, 8, 3, padding=1),    # 7x7x8 -> 7x7x8
            nn.ReLU(),
            nn.Upsample(scale_factor=2),       # 7x7x8 -> 14x14x8
            nn.Conv2d(8, 16, 3, padding=1),   # 14x14x8 -> 14x14x16
            nn.ReLU(),
            nn.Upsample(scale_factor=2),       # 14x14x16 -> 28x28x16
            nn.Conv2d(16, 1, 3, padding=1),   # 28x28x16 -> 28x28x1
            nn.Sigmoid()
        )
    
    def forward(self, x):
        code = self.encoder(x)
        return self.decoder(code)

model = ConvAutoencoder()
print(model)
print(f'\nEncoder output (bottleneck): 7x7x8 = {7*7*8} values')
print(f'Compression: {784}/{7*7*8} = {784/(7*7*8):.1f}x')


---
## Part III: Applications and Decisions


## 9. Autoencoder Applications

| Application | How | Real-World Example |
|:---|:---|:---|
| **Data compression** | Bottleneck = compressed representation | Image/video compression |
| **Feature learning** | Encoder output → downstream classifier | Pre-training with limited labeled data |
| **Denoising** | Denoising AE removes noise | Photo restoration, audio cleanup |
| **Anomaly detection** | Train on normal data; high reconstruction error = anomaly | Fraud detection, manufacturing defects |
| **Colorization** | Grayscale input → color output | Colorizing old black-and-white photos |
| **Generation (VAE)** | Variational autoencoder samples from latent space | Creating synthetic faces, art |

### Extracting the Encoder

After training, you can use just the encoder for feature extraction:
```python
# Use encoder output as features for other tasks
with torch.no_grad():
    features = model.encoder(images)  # Compressed representation
    # Feed 'features' to a classifier, clustering algorithm, etc.
```


## 10. Design Decisions

**Q: How do I choose the bottleneck size?**
Start with 10-20% of input dimension. Too small = lossy (can't reconstruct). Too large = trivial (copies input). Experiment and visualize.

**Q: What loss function?**
- MSE for continuous pixel values
- Binary cross-entropy for pixels in [0,1]
- The choice depends on your data range

**Q: How do I know if it's learning useful features?**
Visualize reconstructions. Close to input but slightly blurry = working. Garbage = bottleneck too small or architecture too simple.

**Q: Autoencoder vs PCA?**
PCA = linear autoencoder. If data has nonlinear structure, autoencoders win. But PCA is faster and has guaranteed optimality for linear projections.

**Q: Dense vs Convolutional autoencoder?**
For images, always convolutional — preserves spatial structure, fewer parameters.


---

## Part III: Generative Adversarial Networks (GANs)


## 11. From Autoencoders to GANs — The Leap to Creation

### The Mental Model

Autoencoders learn to **compress and reconstruct** existing data. But what if we want to **create entirely new data** that looks real? That's what GANs do.

*Analogy:* An autoencoder is like a photocopier — it reproduces what you give it (through a bottleneck). A GAN is like an artist — it creates original works that look like they could be real photographs.

| | Autoencoder | GAN |
|:---|:---|:---|
| **Goal** | Reconstruct input through bottleneck | Generate new data from random noise |
| **Architecture** | Encoder + Decoder | Generator + Discriminator |
| **Loss** | Reconstruction error (pixel-level) | Adversarial (learned by discriminator) |
| **Output quality** | Blurry (averages over possibilities) | Sharp (optimized to fool a critic) |
| **Training data** | Learns to compress X | Learns to generate data that looks like X |

### Generative vs Discriminative Models

| Type | What It Learns | Question It Answers | Example |
|:---|:---|:---|:---|
| **Discriminative** | P(y\|X) — label given data | "What class is this?" | CNN classifier |
| **Generative** | P(X) — distribution of data | "What does realistic data look like?" | GAN, VAE, Diffusion |

**Connection to Generative AI:** GANs (2014) were the first deep learning models that could generate realistic images. The same core idea — "learn the distribution of data, then sample from it" — is behind DALL-E, Midjourney, Stable Diffusion, and the vision components of modern LLMs.


## 12. How GANs Work — The Counterfeiter and the Detective

### The Analogy

Two competing players in an arms race:

**Generator (G) = Counterfeiter:** Takes random noise (blank paper) and tries to produce fake money (images) that looks real. Improves by studying what the detective catches.

**Discriminator (D) = Detective:** Receives both real money and counterfeits, tries to tell them apart. Improves by studying both real and fake samples.

```
Random noise z ──→ [Generator G] ──→ Fake image ──→ [Discriminator D] ──→ Real or Fake?
                                                              ↑
                        Real image from dataset ──────────────┘
```

### The Training Dynamic

1. Detective gets better at spotting fakes → Counterfeiter must improve
2. Counterfeiter produces better fakes → Detective must adapt
3. Eventually, counterfeiter is so good that detective can't tell the difference (D outputs 0.5)
4. **Discard the detective. Keep the counterfeiter.** The generator now produces realistic data.

### The Minimax Objective (The Exam Formula)

```
min_G max_D  V(G,D) = E_x[log D(x)] + E_z[log(1 - D(G(z)))]
```

**What this means in plain English:**
- **D wants to maximize:** high confidence on real images (D(x)→1) + low confidence on fakes (D(G(z))→0)
- **G wants to minimize:** make D(G(z))→1 (fool the discriminator)

**In practice, G maximizes log(D(G(z))) instead** — better gradient signal early in training.


## 13. The GAN Training Loop — Two Networks, Two Optimizers

### The 4-Step Process (Exam-Critical)

```
For each mini-batch:

  Step 1: TRAIN DISCRIMINATOR on real data
    - Feed real images to D, label = 1
    - Loss_real = BCE(D(real), 1)

  Step 2: TRAIN DISCRIMINATOR on fake data
    - Generate fake images: z → G(z)
    - Feed fakes to D, label = 0
    - Loss_fake = BCE(D(G(z)), 0)
    - D_total_loss = Loss_real + Loss_fake
    - Update D's weights (NOT G's weights!)

  Step 3: TRAIN GENERATOR
    - Generate new fake images: z → G(z)
    - Feed fakes to D, but label = 1 (the trick!)
    - G_loss = BCE(D(G(z)), 1)
    - Update G's weights (NOT D's weights!)

  Step 4: REPEAT
```

### Key Details

**Why label fakes as 1 when training G?** The generator wants to fool the discriminator. If D(G(z)) is close to 1, G succeeded. So G's loss measures "how far from fooling D am I?"

**Why `.detach()` when training D on fakes?** When D trains on fake images, we don't want gradients flowing back to update G. `fake_images.detach()` cuts the gradient chain. When G trains, we DO want gradients — no detach.

**Two separate optimizers:** D and G have independent optimizers, often with different learning rates. They're separate networks competing with each other.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# === SIMPLE GAN ARCHITECTURE (MNIST) ===

class Generator(nn.Module):
    """Takes random noise (100 dims) → produces fake image (784 pixels).
    
    Think: blank paper → counterfeiter → fake banknote
    Output: tanh → range [-1, 1] (matching normalized real images)
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(100, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 512),
            nn.LeakyReLU(0.2),
            nn.Linear(512, 784),
            nn.Tanh()              # Output range [-1, 1]
        )
    
    def forward(self, z):
        return self.net(z)

class Discriminator(nn.Module):
    """Takes image (784 pixels) → outputs probability of being real.
    
    Think: banknote → detective → "real or fake?"
    Output: raw logits (BCEWithLogitsLoss applies sigmoid)
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.3),
            nn.Linear(256, 1)      # Single output: real or fake
        )
    
    def forward(self, x):
        return self.net(x.view(-1, 784))

G = Generator()
D = Discriminator()
print(f"Generator params:     {sum(p.numel() for p in G.parameters()):,}")
print(f"Discriminator params: {sum(p.numel() for p in D.parameters()):,}")
print(f"\nGenerator: noise(100) → 256 → 512 → 784 (image)")
print(f"Discriminator: image(784) → 512 → 256 → 1 (real/fake)")


In [ ]:
# === GAN TRAINING LOOP — The 4-Step Pattern ===

loss_fn = nn.BCEWithLogitsLoss()   # Binary cross-entropy (no sigmoid needed)
d_optimizer = optim.Adam(D.parameters(), lr=0.0002)
g_optimizer = optim.Adam(G.parameters(), lr=0.0002)
z_dim = 100  # Latent noise dimension

# One training step (showing the pattern — not a full training loop)
batch_size = 64

# Simulate a batch of real images (normally from DataLoader)
real_images = torch.randn(batch_size, 784)  # Placeholder
real_images = (real_images * 2) - 1          # Scale to [-1, 1]

# === STEP 1-2: Train Discriminator ===
d_optimizer.zero_grad()

# Real images → D should say "real" (label=1)
d_real_pred = D(real_images)
d_real_loss = loss_fn(d_real_pred, torch.ones(batch_size, 1))

# Fake images → D should say "fake" (label=0)
z = torch.randn(batch_size, z_dim)           # Random noise
fake_images = G(z)                            # Generator creates fakes
d_fake_pred = D(fake_images.detach())         # .detach() = don't update G here!
d_fake_loss = loss_fn(d_fake_pred, torch.zeros(batch_size, 1))

d_loss = d_real_loss + d_fake_loss
d_loss.backward()
d_optimizer.step()

# === STEP 3: Train Generator ===
g_optimizer.zero_grad()

z = torch.randn(batch_size, z_dim)
fake_images = G(z)
d_fake_pred = D(fake_images)                  # NO detach — we want gradients for G!
g_loss = loss_fn(d_fake_pred, torch.ones(batch_size, 1))  # Label fakes as REAL (the trick!)
g_loss.backward()
g_optimizer.step()

print(f"D loss: {d_loss.item():.4f} (wants to tell real from fake)")
print(f"G loss: {g_loss.item():.4f} (wants to fool D)")
print(f"\nKey: D trains with .detach() on fakes. G trains WITHOUT .detach().")


## 14. SimpleGAN vs DCGAN — The Architecture Difference

### The Mental Model

**SimpleGAN** uses fully connected layers — it flattens images to vectors, losing spatial structure. Results are blurry.

**DCGAN (Deep Convolutional GAN)** uses convolutional layers — preserves spatial structure, produces sharper images.

| Feature | SimpleGAN | DCGAN |
|:---|:---|:---|
| **Generator layers** | Dense (Linear) | ConvTranspose2d (upsampling) |
| **Discriminator layers** | Dense (Linear) | Conv2d (downsampling) |
| **Image quality** | Blurry | Sharp |
| **Key G layer** | `nn.Linear(100, 784)` | `nn.ConvTranspose2d(100, 128, 4, 1, 0)` |
| **Key D layer** | `nn.Linear(784, 512)` | `nn.Conv2d(1, 32, 4, 2, 1)` |
| **Normalization** | Dropout | BatchNorm |
| **G output activation** | Tanh | Tanh |
| **D output activation** | None (logits) or Sigmoid | Sigmoid |

### ConvTranspose2d — The Upsampling Layer

**Conv2d** goes from large → small (downsampling). Used in discriminator and regular CNNs.

**ConvTranspose2d** goes from small → large (upsampling). Used in generator — the **reverse** of convolution.

```
Generator path:
  Noise (100,1,1) → ConvTranspose → (128,4,4) → ConvTranspose → (64,8,8) → ConvTranspose → (1,28,28)
  Small latent code → progressively larger feature maps → full image

Discriminator path (reverse):
  Image (1,28,28) → Conv → (32,14,14) → Conv → (64,7,7) → Conv → (1,1,1) → Real/Fake
  Full image → progressively smaller feature maps → single decision
```

### DCGAN Design Rules (from the original paper)

1. Replace pooling with **strided convolutions** (D) and **strided transposed convolutions** (G)
2. Use **BatchNorm** in both G and D (except G output and D input)
3. **No fully connected layers** — all convolutional
4. **ReLU** in G (Tanh on output), **LeakyReLU** in D
5. **Adam optimizer** with lr=0.0002


## 15. All Possible Questions About GANs

**Q: Why not just use an autoencoder to generate images?**
Autoencoders optimize pixel-level reconstruction (MSE) → produces blurry averages. GANs optimize "fooling the discriminator" → captures high-level realism, sharper edges, more natural textures.

**Q: What makes GAN training unstable?**
The two networks are in a delicate balance. If D becomes too strong → G gets no useful gradient (all fakes rejected). If G becomes too strong → D can't learn. This is why GANs are notoriously difficult to train.

**Q: What is "mode collapse"?**
When G learns to produce only one or a few types of outputs that fool D, ignoring diversity. E.g., generating only the digit "1" because D can't tell those fakes from real 1s.

**Q: What's the difference between BCELoss and BCEWithLogitsLoss?**
- `BCELoss`: expects sigmoid-activated input [0,1]. D must have sigmoid on output.
- `BCEWithLogitsLoss`: combines sigmoid + BCE in one numerically stable operation. D outputs raw logits (no sigmoid). **Preferred in practice.**

**Q: After training, what do you keep?**
**Only the Generator.** The Discriminator was just a training tool — discard it. Use G(z) to generate new images from random noise.

**Q: What are GANs used for in the real world?**

| Application | Example |
|:---|:---|
| Image synthesis | StyleGAN (thispersondoesnotexist.com) |
| Image-to-image | Pix2Pix (sketch → photo), CycleGAN (horse → zebra) |
| Super-resolution | ESRGAN (enhance old photos/video) |
| Data augmentation | Generate synthetic medical images for rare diseases |
| Text-to-image | Early DALL-E concepts (now diffusion-based) |


---

## Worked Exam Problems — Autoencoders

### Problem 1: "Trace values through a simple autoencoder"

**Architecture:** Input(4) → Encoder(2, ReLU) → Decoder(4, Sigmoid)

**Encoder weights and biases:**
```
W_enc = [[0.5, 0.3, -0.2, 0.4],     b_enc = [0.1, -0.1]
         [0.1, -0.5, 0.6, 0.2]]
```

**Input:** x = [1.0, 0.0, 1.0, 0.0]

**Step 1: Encoder forward pass (net input)**
```
z1 = 0.5*1.0 + 0.3*0.0 + (-0.2)*1.0 + 0.4*0.0 + 0.1 = 0.5 - 0.2 + 0.1 = 0.4
z2 = 0.1*1.0 + (-0.5)*0.0 + 0.6*1.0 + 0.2*0.0 + (-0.1) = 0.1 + 0.6 - 0.1 = 0.6
```

**Step 2: ReLU activation**
```
h = ReLU([0.4, 0.6]) = [0.4, 0.6]    ← This is the CODE (bottleneck)
```

Compression: 4 dimensions → 2 dimensions (2x compression)

**Step 3: Decoder would map [0.4, 0.6] back to 4 dimensions**
(Decoder weights not shown — the network learns these during training)

**Step 4: Loss = MSE between input [1.0, 0.0, 1.0, 0.0] and reconstruction**

**Key insight:** The bottleneck [0.4, 0.6] must encode enough information about [1.0, 0.0, 1.0, 0.0] for the decoder to reconstruct it. After training, similar inputs produce similar bottleneck codes.

### Problem 2: "Calculate compression ratio and parameter count"

**Architecture:** Input(784) → Dense(256, ReLU) → Dense(64, ReLU) → Dense(256, ReLU) → Dense(784, Sigmoid)

**Compression ratio:**
```
Input dimension:      784
Bottleneck dimension: 64
Compression ratio:    784 / 64 = 12.25x
```

**Parameter count:**
```
Encoder:
  Dense(784→256): (784 + 1) * 256 = 200,960
  Dense(256→64):  (256 + 1) * 64  = 16,448

Decoder:
  Dense(64→256):  (64 + 1) * 256  = 16,640
  Dense(256→784): (256 + 1) * 784 = 201,488

Total: 200,960 + 16,448 + 16,640 + 201,488 = 435,536 parameters
```

**Notice:** Encoder and decoder have nearly the same parameter count (mirror architecture). The bottleneck layers have the fewest parameters.

### Problem 3: "What makes a denoising autoencoder different?"

**Standard autoencoder:**
- Input: clean image x
- Target: clean image x
- Loss: L(x, g(f(x)))

**Denoising autoencoder:**
- Input: noisy image x̃ = x + noise
- Target: **clean** image x (NOT x̃!)
- Loss: L(x, g(f(x̃)))

**Why this matters:** The standard autoencoder can learn the identity function (just copy). The denoising autoencoder CANNOT copy — the noise is different every time. It must learn the underlying structure to "see through" the noise.

### Problem 4: "Convolutional autoencoder — trace the shapes"

**Architecture:**
```
Input:     28x28x1
Conv(1→16, 3x3, padding=1):  28x28x16    params: (3*3*1+1)*16 = 160
MaxPool(2x2):                14x14x16    params: 0
Conv(16→8, 3x3, padding=1):  14x14x8     params: (3*3*16+1)*8 = 1,160
MaxPool(2x2):                7x7x8       params: 0

BOTTLENECK: 7x7x8 = 392 values (vs 784 input = 2x compression)

Conv(8→8, 3x3, padding=1):   7x7x8      params: (3*3*8+1)*8 = 584
UpSample(2x):                14x14x8     params: 0
Conv(8→16, 3x3, padding=1):  14x14x16    params: (3*3*8+1)*16 = 1,168
UpSample(2x):                28x28x16    params: 0
Conv(16→1, 3x3, padding=1):  28x28x1     params: (3*3*16+1)*1 = 145

Total parameters: 160 + 1,160 + 584 + 1,168 + 145 = 3,217
```

**Key observations:**
- UpSampling2d has 0 parameters (just duplicates pixels — like reverse pooling)
- The encoder and decoder are roughly symmetric
- Conv autoencoder (3,217 params) is much smaller than dense autoencoder (435,536 params) for the same image


### Problem 5: "Describe the GAN training process"

**Step 1:** Sample real images from training data, assign label = 1 (real)

**Step 2:** Generate fake images by feeding random noise z through Generator: G(z). Assign label = 0 (fake)

**Step 3:** Train Discriminator:
- Feed real images → D should output close to 1
- Feed fake images (detached from G) → D should output close to 0
- D_loss = BCE(D(real), 1) + BCE(D(fake), 0)
- Update D's weights only

**Step 4:** Train Generator:
- Generate new fakes: G(z)
- Feed through D, but label as 1 (trick!)
- G_loss = BCE(D(G(z)), 1)
- Update G's weights only

**Key detail:** When training D, use `.detach()` on fake images (don't update G). When training G, don't detach (need gradients to flow through D back to G).

### Problem 6: "What is the GAN objective function? Explain each term."

```
min_G max_D  V(G,D) = E_x[log D(x)] + E_z[log(1 - D(G(z)))]
```

- **E_x[log D(x)]:** Expected log-probability that D correctly identifies real data as real. D wants this HIGH.
- **E_z[log(1 - D(G(z)))]:** Expected log-probability that D correctly identifies fake data as fake. D wants this HIGH (1 - D(G(z)) close to 1). G wants this LOW (D(G(z)) close to 1, making 1-D(G(z)) close to 0).
- **D maximizes** the whole expression (be a good detective)
- **G minimizes** the whole expression (be a good counterfeiter)

### Problem 7: "SimpleGAN vs DCGAN — what are the key differences?"

| | SimpleGAN | DCGAN |
|:---|:---|:---|
| Generator | Fully connected (Dense) layers | ConvTranspose2d (learned upsampling) |
| Discriminator | Fully connected (Dense) layers | Conv2d (learned downsampling) |
| Spatial structure | Lost (images flattened to vectors) | Preserved (2D throughout) |
| Image quality | Blurry | Sharp |
| Normalization | Dropout | BatchNorm |
| Why DCGAN is better | FC layers lose spatial relationships between pixels | Conv layers maintain neighborhood structure, weight sharing reduces parameters |


---
## 11. Self-Check and Exam Practice

### Conceptual Questions

1. What is an autoencoder? What makes it 'unsupervised'?
2. What is the bottleneck and why is it necessary?
3. What is the difference between undercomplete and overcomplete autoencoders?
4. How does a denoising autoencoder differ from a standard autoencoder? What is the loss compared against?
5. Why use a convolutional autoencoder instead of a dense one for images?
6. Name three real-world applications of autoencoders.
7. What is the relationship between autoencoders and PCA?
8. You train an autoencoder with bottleneck=784 on 784-dimensional input. What happens? Why?

<details>
<summary>Click to reveal answers</summary>

1. A network that learns to reconstruct its input through a bottleneck. Unsupervised because Y=X — no external labels needed.
2. The compressed representation between encoder and decoder. Forces the network to learn which features matter, preventing trivial identity copying.
3. Undercomplete: bottleneck < input → forced compression. Overcomplete: bottleneck >= input → can cheat by copying. Must add regularization.
4. Denoising AE receives corrupted input but loss compares output to CLEAN original. Forces learning of robust structure, not noise.
5. Dense flattens images, destroying spatial structure. Conv preserves 2D layout and uses weight sharing for fewer parameters.
6. Data compression, denoising, anomaly detection, feature learning, colorization, generation (VAEs).
7. A linear autoencoder (no activations) learns the same subspace as PCA — directions of maximum variance.
8. It can learn the identity function (copy input to output) without learning useful features. The bottleneck is too large — no compression forced.

</details>


### GAN Questions

9. What is the role of the Generator? What is the role of the Discriminator?
10. Write the GAN minimax objective function and explain what each term means.
11. In the training loop, why do we use `.detach()` when training the Discriminator on fake images?
12. What is mode collapse?
13. After training is complete, which network do you keep — Generator or Discriminator?
14. What is ConvTranspose2d and why does the Generator use it?
15. Name two advantages of DCGAN over SimpleGAN.

<details>
<summary>Click to reveal GAN answers</summary>

9. Generator: creates fake data from random noise. Discriminator: classifies inputs as real or fake.

10. min_G max_D V(G,D) = E_x[log D(x)] + E_z[log(1-D(G(z)))]. First term: D's ability to recognize real data. Second term: D's ability to reject fakes (which G tries to undermine).

11. Because we only want to update D's weights, not G's. If we don't detach, gradients flow back through G and update its weights — which would undermine the alternating training pattern.

12. When the Generator learns to produce only one or a few outputs that fool D, ignoring the diversity of real data. E.g., only generating the digit "1."

13. Keep the Generator. The Discriminator was only a training tool — its job is done.

14. ConvTranspose2d is "reverse convolution" — it upsamples from small feature maps to large ones. The Generator needs it because it starts from a tiny noise vector and must produce a full-sized image.

15. (a) DCGAN preserves spatial structure (conv layers maintain 2D layout, FC layers flatten it). (b) DCGAN produces sharper images because convolutions respect pixel neighborhoods.

</details>
